# Experiment D: Hybrid ELO + Pi-Ratings GLM

**Hypothesis:** ELO and pi-ratings capture different aspects of team strength: ELO encodes relative win probability over long history, while pi-ratings encode recent score-margin (goal difference) and separate home/away attacking/defensive strength. Using both together should be strictly better than either alone.

**Setup:**
- Features: `home_elo`, `away_elo`, `elo_diff` (ELO) + `pi_home`, `pi_away`, `pi_diff` (pi-ratings) + same form features as notebooks 11 and 17
- Same 4-fold LOTO-CV over WC 2010/2014/2018/2022
- Pi-ratings computed from all matches before the WC year cutoff (no leakage)
- Same shared Poisson GLM with stacked home/away rows, `alpha=0.1`

**Baselines:**
- Autofill (FIFA-ranking): 4.137 pts/match
- ELO Poisson GLM (nb 11): 4.336 pts/match
- Pi-Ratings GLM (nb 17):  4.375 pts/match
- Exp B Reg GLM+dummies (nb 15): 4.379 pts/match

In [1]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import json
from functools import lru_cache
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from pathlib import Path

from penaltyblog.ratings import PiRatingSystem

from evaluation.sporza import score_breakdown, sporza_points_series

DATA = Path('../../data/processed')
INTERIM = Path('../../data/interim')
WC_YEARS = [2010, 2014, 2018, 2022]
MAX_GOALS = 8

# Hybrid feature set: ELO + pi-ratings + form
HOME_FEATURES = [
    'home_elo', 'away_elo', 'elo_diff',
    'pi_home', 'pi_away', 'pi_diff',
    'is_neutral',
    'home_form_wr', 'away_form_wr',
    'home_form_gf', 'away_form_gf',
    'home_form_ga', 'away_form_ga',
    'tournament_weight',
]
AWAY_FEATURES = [
    'away_elo', 'home_elo', 'elo_diff',
    'pi_away', 'pi_home', 'pi_diff',
    'is_neutral',
    'away_form_wr', 'home_form_wr',
    'away_form_gf', 'home_form_gf',
    'away_form_ga', 'home_form_ga',
    'tournament_weight',
]

manifest = json.loads((DATA / 'split_manifest.json').read_text())
autofill_mean = manifest['autofill_baseline']['pooled_mean_pts']
print(f'Autofill baseline: {autofill_mean:.3f} pts/match')
print(f'HOME_FEATURES ({len(HOME_FEATURES)}): {HOME_FEATURES}')

Autofill baseline: 4.137 pts/match
HOME_FEATURES (14): ['home_elo', 'away_elo', 'elo_diff', 'pi_home', 'pi_away', 'pi_diff', 'is_neutral', 'home_form_wr', 'away_form_wr', 'home_form_gf', 'away_form_gf', 'home_form_ga', 'away_form_ga', 'tournament_weight']


In [2]:
def bootstrap_ci(pts, n_boot=10000):
    rng = np.random.default_rng(42)
    vals = pts.values
    boot = np.array([rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {'mean': float(vals.mean()), 'ci_lo': float(lo), 'ci_hi': float(hi)}


def permutation_test(pts_a, pts_b, n_perm=10000):
    """Paired permutation test: H0 = mean(a) <= mean(b). Returns p-value."""
    rng = np.random.default_rng(42)
    obs_diff = pts_a.mean() - pts_b.mean()
    diffs = np.zeros(n_perm)
    for i in range(n_perm):
        swap = rng.integers(0, 2, size=len(pts_a)).astype(bool)
        a = np.where(swap, pts_b, pts_a)
        b = np.where(swap, pts_a, pts_b)
        diffs[i] = a.mean() - b.mean()
    return float((diffs >= obs_diff).mean())


def build_X(df, features):
    X = df[features].copy().fillna(df[features].median())
    return X.values


def build_pipe(alpha=0.1):
    return Pipeline([('scaler', StandardScaler()),
                     ('poisson', PoissonRegressor(alpha=alpha, max_iter=300))])


@lru_cache(maxsize=50000)
def best_score_cached(lh_r, la_r):
    """Score (h, a) maximising expected Sporza pts given Poisson(lh) x Poisson(la)."""
    goals = np.arange(MAX_GOALS + 1)
    ph_pmf = poisson.pmf(goals, lh_r)
    pa_pmf = poisson.pmf(goals, la_r)
    joint = np.outer(ph_pmf, pa_pmf)

    def sporza_mat(pred_h, pred_a):
        pts = 0.0
        for ah in range(MAX_GOALS + 1):
            for aa in range(MAX_GOALS + 1):
                p = joint[ah, aa]
                if p < 1e-9:
                    continue
                if pred_h == ah and pred_a == aa:
                    pts += p * 10
                elif (pred_h - pred_a) == (ah - aa):
                    pts += p * 7
                elif np.sign(pred_h - pred_a) == np.sign(ah - aa):
                    pts += p * 5
                else:
                    pts += p * 1
        return pts

    best_pts, best_h, best_a = -1.0, 1, 0
    for ph_idx in range(MAX_GOALS + 1):
        for pa_idx in range(MAX_GOALS + 1):
            ep = sporza_mat(ph_idx, pa_idx)
            if ep > best_pts:
                best_pts, best_h, best_a = ep, ph_idx, pa_idx
    return best_h, best_a

print('Helper functions defined.')

Helper functions defined.


In [3]:
WC_CUTOFFS = {
    2010: '2010-06-01',
    2014: '2014-06-01',
    2018: '2018-06-01',
    2022: '2022-11-01',
}


def compute_pi_ratings_before_cutoff(matches, cutoff_date):
    pi = PiRatingSystem(alpha=0.15, beta=0.10, k=0.75)
    training_matches = matches[matches['date'] < cutoff_date].sort_values('date')
    for _, row in training_matches.iterrows():
        goal_diff = int(row['home_score'] - row['away_score'])
        pi.update_ratings(row['home_team'], row['away_team'], goal_diff)
    print(f'  Cutoff {cutoff_date}: {len(training_matches)} matches, {len(pi.team_ratings)} teams rated')
    return pi.team_ratings


def add_pi_features(df, team_ratings):
    df = df.copy()
    df['pi_home'] = df['home_team'].map(lambda t: team_ratings[t]['home'] if t in team_ratings else 0.0)
    df['pi_away'] = df['away_team'].map(lambda t: team_ratings[t]['away'] if t in team_ratings else 0.0)
    df['pi_diff'] = df['pi_home'] - df['pi_away']
    return df


matches_full = pd.read_parquet(INTERIM / 'matches.parquet')
print(f'Full match history: {len(matches_full)} matches '
      f'({matches_full.date.min().date()} to {matches_full.date.max().date()})')

print('\nComputing pi-ratings for each fold cutoff...')
pi_ratings_by_year = {}
for year, cutoff in WC_CUTOFFS.items():
    pi_ratings_by_year[year] = compute_pi_ratings_before_cutoff(matches_full, cutoff)
print('Done.')

Full match history: 32288 matches (1990-01-12 to 2026-06-11)

Computing pi-ratings for each fold cutoff...


  Cutoff 2010-06-01: 16704 matches, 275 teams rated


  Cutoff 2014-06-01: 20658 matches, 294 teams rated


  Cutoff 2018-06-01: 24399 matches, 312 teams rated


  Cutoff 2022-11-01: 28505 matches, 323 teams rated
Done.


In [4]:
# LOTO-CV: Hybrid ELO + Pi-Ratings GLM
all_pts = []
fold_results = []
fold_pipes = {}

for year in WC_YEARS:
    train = pd.read_parquet(DATA / f'train_fold_{year}.parquet')
    evalf = pd.read_parquet(DATA / f'eval_fold_{year}.parquet')

    ratings = pi_ratings_by_year[year]
    train = add_pi_features(train, ratings)
    evalf = add_pi_features(evalf, ratings)

    X_tr = np.vstack([build_X(train, HOME_FEATURES), build_X(train, AWAY_FEATURES)])
    y_tr = np.concatenate([train['home_score'].values, train['away_score'].values])

    pipe = build_pipe(alpha=0.1)
    pipe.fit(X_tr, y_tr)
    fold_pipes[year] = pipe

    lh = pipe.predict(build_X(evalf, HOME_FEATURES))
    la = pipe.predict(build_X(evalf, AWAY_FEATURES))

    preds = [best_score_cached(round(float(h), 2), round(float(a), 2)) for h, a in zip(lh, la)]
    pred_home = pd.Series([p[0] for p in preds])
    pred_away = pd.Series([p[1] for p in preds])

    bd = score_breakdown(pred_home, pred_away,
                         evalf['home_score'].reset_index(drop=True),
                         evalf['away_score'].reset_index(drop=True))
    pts = sporza_points_series(pred_home, pred_away,
                               evalf['home_score'].reset_index(drop=True),
                               evalf['away_score'].reset_index(drop=True))
    all_pts.extend(pts.tolist())
    fold_results.append({'year': year, 'mean_pts': bd['mean_pts'],
                         'pct_exact': bd['pct_exact'], 'pct_correct_result': bd['pct_correct_result']})
    print(f'WC {year}: mean_pts={bd["mean_pts"]:.3f}  '
          f'exact={bd["pct_exact"]:.1%}  correct={bd["pct_correct_result"]:.1%}')

pooled_pts = pd.Series(all_pts)
ci = bootstrap_ci(pooled_pts)
print(f'\nPooled LOTO-CV ({len(pooled_pts)} matches): {ci["mean"]:.3f} pts/match  '
      f'95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]')

WC 2010: mean_pts=4.266  exact=20.3%  correct=51.6%


WC 2014: mean_pts=5.031  exact=18.8%  correct=65.6%


WC 2018: mean_pts=4.797  exact=20.3%  correct=60.9%
WC 2022: mean_pts=3.453  exact=7.8%  correct=48.4%



Pooled LOTO-CV (256 matches): 4.387 pts/match  95% CI [3.988, 4.789]


In [5]:
# Comparison table and permutation tests vs ELO GLM and pi-ratings GLM

# Re-run ELO-only GLM to get paired pts series
ELO_HOME = ['home_elo', 'away_elo', 'elo_diff', 'is_neutral',
            'home_form_wr', 'away_form_wr', 'home_form_gf', 'away_form_gf',
            'home_form_ga', 'away_form_ga', 'tournament_weight']
ELO_AWAY = ['away_elo', 'home_elo', 'elo_diff', 'is_neutral',
            'away_form_wr', 'home_form_wr', 'away_form_gf', 'home_form_gf',
            'away_form_ga', 'home_form_ga', 'tournament_weight']
PI_HOME  = ['pi_home', 'pi_away', 'pi_diff', 'is_neutral',
            'home_form_wr', 'away_form_wr', 'home_form_gf', 'away_form_gf',
            'home_form_ga', 'away_form_ga', 'tournament_weight']
PI_AWAY  = ['pi_away', 'pi_home', 'pi_diff', 'is_neutral',
            'away_form_wr', 'home_form_wr', 'away_form_gf', 'home_form_gf',
            'away_form_ga', 'home_form_ga', 'tournament_weight']

elo_all_pts, pi_all_pts = [], []

for year in WC_YEARS:
    train = pd.read_parquet(DATA / f'train_fold_{year}.parquet')
    evalf = pd.read_parquet(DATA / f'eval_fold_{year}.parquet')
    ratings = pi_ratings_by_year[year]
    train_pi = add_pi_features(train, ratings)
    evalf_pi = add_pi_features(evalf, ratings)

    for feat_h, feat_a, store in [(ELO_HOME, ELO_AWAY, elo_all_pts),
                                   (PI_HOME, PI_AWAY, pi_all_pts)]:
        df_tr = train_pi if feat_h == PI_HOME else train
        df_ev = evalf_pi if feat_h == PI_HOME else evalf
        pipe = build_pipe(alpha=0.1)
        pipe.fit(np.vstack([build_X(df_tr, feat_h), build_X(df_tr, feat_a)]),
                 np.concatenate([df_tr['home_score'].values, df_tr['away_score'].values]))
        lh = pipe.predict(build_X(df_ev, feat_h))
        la = pipe.predict(build_X(df_ev, feat_a))
        preds = [best_score_cached(round(float(h), 2), round(float(a), 2)) for h, a in zip(lh, la)]
        pts = sporza_points_series(
            pd.Series([p[0] for p in preds]), pd.Series([p[1] for p in preds]),
            df_ev['home_score'].reset_index(drop=True),
            df_ev['away_score'].reset_index(drop=True))
        store.extend(pts.tolist())

hybrid_pts = np.array(all_pts)
elo_pts    = np.array(elo_all_pts)
pi_pts     = np.array(pi_all_pts)

p_vs_elo = permutation_test(hybrid_pts, elo_pts)
p_vs_pi  = permutation_test(hybrid_pts, pi_pts)

ELO_MEAN = float(np.mean(elo_pts))
PI_MEAN  = float(np.mean(pi_pts))

print('Per-fold results:')
print(f'  {"Fold":<8} {"Hybrid":>8} {"ELO-GLM":>9} {"Pi-GLM":>9}')
ELO_FOLD = {2010: None, 2014: None, 2018: None, 2022: None}
PI_FOLD  = {2010: None, 2014: None, 2018: None, 2022: None}
for i, year in enumerate(WC_YEARS):
    ELO_FOLD[year] = float(np.mean(elo_all_pts[i*64:(i+1)*64]))
    PI_FOLD[year]  = float(np.mean(pi_all_pts[i*64:(i+1)*64]))
    print(f'  WC {year}   {fold_results[i]["mean_pts"]:>8.3f} {ELO_FOLD[year]:>9.3f} {PI_FOLD[year]:>9.3f}')

print(f'\nPooled:')
print(f'  Hybrid ELO+Pi:      {ci["mean"]:.3f}  95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]')
print(f'  ELO GLM:            {ELO_MEAN:.3f}')
print(f'  Pi-Ratings GLM:     {PI_MEAN:.3f}')
print(f'  Autofill:           {autofill_mean:.3f}')
print(f'\nDelta vs ELO:        {ci["mean"] - ELO_MEAN:+.3f}')
print(f'Delta vs Pi-ratings: {ci["mean"] - PI_MEAN:+.3f}')
print(f'Delta vs autofill:   {ci["mean"] - autofill_mean:+.3f}')
print(f'\nPermutation test (Hybrid > ELO):        p = {p_vs_elo:.4f}')
print(f'Permutation test (Hybrid > Pi-ratings): p = {p_vs_pi:.4f}')

Per-fold results:
  Fold       Hybrid   ELO-GLM    Pi-GLM
  WC 2010      4.266     4.141     4.266
  WC 2014      5.031     4.875     4.969
  WC 2018      4.797     4.562     4.797
  WC 2022      3.453     3.766     3.469

Pooled:
  Hybrid ELO+Pi:      4.387  95% CI [3.988, 4.789]
  ELO GLM:            4.336
  Pi-Ratings GLM:     4.375
  Autofill:           4.137

Delta vs ELO:        +0.051
Delta vs Pi-ratings: +0.012
Delta vs autofill:   +0.250

Permutation test (Hybrid > ELO):        p = 0.3662
Permutation test (Hybrid > Pi-ratings): p = 0.4383


In [6]:
# Feature importance via coefficient inspection
import warnings
warnings.filterwarnings('ignore')

# Use WC 2022 fold pipe (largest training set)
pipe_2022 = fold_pipes[2022]
scaler = pipe_2022.named_steps['scaler']
model  = pipe_2022.named_steps['poisson']

coef_df = pd.DataFrame({
    'feature': HOME_FEATURES,
    'coef': model.coef_,
    'abs_coef': np.abs(model.coef_),
}).sort_values('abs_coef', ascending=False)

print('Feature coefficients (WC 2022 fold, home-goals model):')
print(coef_df.to_string(index=False))

print('\nELO vs pi-rating group contribution:')
elo_features = [f for f in HOME_FEATURES if 'elo' in f]
pi_features  = [f for f in HOME_FEATURES if 'pi_' in f]
form_features = [f for f in HOME_FEATURES if 'form' in f]
print(f'  ELO features  ({len(elo_features)}): sum |coef| = {coef_df[coef_df.feature.isin(elo_features)]["abs_coef"].sum():.4f}')
print(f'  Pi features   ({len(pi_features)}): sum |coef| = {coef_df[coef_df.feature.isin(pi_features)]["abs_coef"].sum():.4f}')
print(f'  Form features ({len(form_features)}): sum |coef| = {coef_df[coef_df.feature.isin(form_features)]["abs_coef"].sum():.4f}')

Feature coefficients (WC 2022 fold, home-goals model):
          feature      coef  abs_coef
          pi_away -0.305807  0.305807
          pi_home  0.294001  0.294001
         home_elo  0.142158  0.142158
         away_elo -0.124523  0.124523
     away_form_ga  0.103856  0.103856
     home_form_gf  0.040507  0.040507
       is_neutral  0.015441  0.015441
          pi_diff  0.013171  0.013171
     away_form_gf  0.012654  0.012654
     home_form_ga -0.010537  0.010537
         elo_diff  0.006520  0.006520
tournament_weight -0.004668  0.004668
     away_form_wr -0.001846  0.001846
     home_form_wr  0.001817  0.001817

ELO vs pi-rating group contribution:
  ELO features  (3): sum |coef| = 0.2732
  Pi features   (3): sum |coef| = 0.6130
  Form features (6): sum |coef| = 0.1712


In [7]:
# Log to MLflow
mlflow.set_tracking_uri('sqlite:////Users/seppe.vanswegenoven/projects/wk_2026_match_predictor/mlflow.db')
mlflow.set_experiment('wk2026-tabular-2026-06-14')

with mlflow.start_run(run_name='hybrid_elo_pi_glm_loto') as run:
    mlflow.set_tags({
        'modality': 'tabular',
        'approach': 'poisson_glm_hybrid_elo_pi',
        'eval_strategy': 'loto_cv',
        'dataset_version': 'split_v2',
        'experiment': 'D',
        'hypothesis': 'elo_and_pi_ratings_complementary',
    })
    mlflow.log_params({
        'home_features': ','.join(HOME_FEATURES),
        'alpha': 0.1,
        'score_strategy': 'max_expected_sporza_pts',
        'max_goals_grid': MAX_GOALS,
        'pi_alpha': 0.15,
        'pi_beta': 0.10,
        'pi_k': 0.75,
        'elo_glm_baseline': ELO_MEAN,
        'pi_glm_baseline': PI_MEAN,
    })
    mlflow.log_metric('loto_mean_sporza_pts', ci['mean'])
    mlflow.log_metric('loto_ci_lo', ci['ci_lo'])
    mlflow.log_metric('loto_ci_hi', ci['ci_hi'])
    mlflow.log_metric('autofill_baseline', autofill_mean)
    mlflow.log_metric('delta_vs_autofill', ci['mean'] - autofill_mean)
    mlflow.log_metric('delta_vs_elo_glm', ci['mean'] - ELO_MEAN)
    mlflow.log_metric('delta_vs_pi_glm', ci['mean'] - PI_MEAN)
    mlflow.log_metric('perm_test_p_vs_elo', p_vs_elo)
    mlflow.log_metric('perm_test_p_vs_pi', p_vs_pi)
    for r in fold_results:
        mlflow.log_metric(f'fold_{r["year"]}_mean_pts', r['mean_pts'])
        mlflow.log_metric(f'fold_{r["year"]}_pct_exact', r['pct_exact'])
        mlflow.log_metric(f'fold_{r["year"]}_pct_correct', r['pct_correct_result'])
    mlflow.sklearn.log_model(fold_pipes[2022], 'hybrid_glm_wc2022_fold')
    run_id = run.info.run_id

print(f'MLflow run_id: {run_id}')
print(f'\nFinal: {ci["mean"]:.3f} pts/match  95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]')

2026/06/14 17:48:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/06/14 17:48:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/06/14 17:48:37 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


MLflow run_id: 38c71747498b426aa0bca45a05c0a81b

Final: 4.387 pts/match  95% CI [3.988, 4.789]
